<a href="https://colab.research.google.com/github/gagan-bajwa/meridian/blob/main/demo/Meridian_RF_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google/meridian/blob/main/demo/Meridian_RF_Demo.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google/meridian/blob/main/demo/Meridian_RF_Demo.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

# **Meridian Reach and Frequency Demo**

Welcome to the Meridian end-to-end demo for Reach and Frequency data. This simplified demo showcases the fundamental functionalities and basic usage of the library on data containing Reach and Frequency channels, including working examples of the major modeling steps:


<ol start="0">
  <li><a href="#install">Install</a></li>
  <li><a href="#load-data">Load the data</a></li>
  <li><a href="#configure-model">Configure the model</a></li>
  <li><a href="#model-diagnostics">Run model diagnostics</a></li>
  <li><a href="#generate-summary">Generate model results & two-page output</a></li>
  <li><a href="#generate-optimize">Run budget optimization & two-page output</a></li>
  <li><a href="#save-model">Save the model object</a></li>
</ol>


Note that this notebook skips all of the exploratory data analysis and preprocessing steps. It assumes that you have completed these tasks before reaching this point in the demo.

This notebook utilizes sample data. As a result, the numbers and results obtained might not accurately reflect what you encounter when working with a real dataset.

<a name="install"></a>
## Step 0: Install

1\. Make sure you are using one of the available GPU Colab runtimes which is **required** to run Meridian. You can change your notebook's runtime in `Runtime > Change runtime type` in the menu. All users can use the T4 GPU runtime which is sufficient to run the demo colab, free of charge. Users who have purchased one of Colab's paid plans have access to premium GPUs (such as V100, A100 or L4 Nvidia GPU).

2\. Install the latest version of Meridian, and verify that GPU is available.

In [ ]:
# Install meridian: from PyPI @ latest release
!pip install --upgrade google-meridian[colab,and-cuda]

# Install meridian: from PyPI @ specific version
# !pip install google-meridian[colab,and-cuda]==1.0.3

# Install meridian: from GitHub @HEAD
# !pip install --upgrade "google-meridian[colab,and-cuda] @ git+https://github.com/google/meridian.git"

In [ ]:
import IPython
from meridian import constants
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

In [ ]:
# @markdown If you are using Colab Free, Colab Pro, run this cell to mount your Google Drive.
from google.colab import drive
drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)
subfolder = '' # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# Change this "MyDrive" to other share folders name if you would like to use a different drive.
meridian_root = f'{drive_mount}/MyDrive/{subfolder}'
is_enterprise_user=False

<a name="load-data"></a>
## Step 1: Load the data

Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/geo_media_rf.csv) as follows.

1\. Read the data into a Pandas DataFrame.

In [ ]:
df = pd.read_csv('/content/tapi_brand_non_brand_without_nans.csv'
    # Optionally, use `f"${meridian_root}/<path_to_csv>"` to load data from the mounted storage.
    #"https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_all_channels.csv"
)

In [ ]:
df.head()


In [ ]:
# change week column name to time
df = df.rename(columns={'week': 'time'})

2\. Create a DataFrameInputDataBuilder instance.

In [ ]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='revenue',
    default_kpi_column='revenue',
    #default_time_column='week'
   # default_revenue_per_kpi_column='revenue_per_conversion',
)

3\. Offer the components to the builder. Note that the components may be offered all at once or piecewise.
If your data includes organic media or non-media treatments, you can add them using `with_organic_media` and `with_non_media_treatments` methods. For the definition of each variable, see
[Collect and organize your data](https://developers.google.com/meridian/docs/user-guide/collect-data)

In [ ]:
builder = (
    builder.with_kpi(df)

    #.with_revenue_per_kpi(df)
    #.with_population(df)
    #.with_controls(
        #df, control_cols=["sentiment_score_control", "competitor_sales_control"]
    #)
)
channels = ['email_YT_Tiktok',
 'TV_SVOD_Nat_TV',
 'DOOH_Programmatic_Audio_DD_Leaflets',
 'bing_brand',
 'bing_non_brand',
 'google_brand',
 'google_non_brand',
 #'paid_social_prospecting',
 #'paid_social_remarketing'
            ]
builder = builder.with_media(
    df,
    media_cols=['clicks_email_YT_Tiktok',
                'impressions_TV_SVOD_Nat_TV',
                'impressions_DOOH_Programmatic_Audio_DD_Leaflets',
                'bing_brand_clicks',
                'bing_non_brand_clicks',
                'google_brand_clicks',
                'google_non_brand_clicks',
                #'paid_social_prospecting_impressions',
              #'paid_social_remarketing_impressions'
                ],
    media_spend_cols=['spend_email_YT_Tiktok',
 'spend_TV_SVOD_Nat_TV',
 'spend_DOOH_Programmatic_Audio_DD_Leaflets',
 'bing_brand_spend',
 'bing_non_brand_spend',
 'google_brand_spend',
 'google_non_brand_spend',
# 'paid_social_prospecting_spend',
 #'paid_social_remarketing_spend'
                      ],
    media_channels=channels,
).with_reach(
    df,
    reach_cols=['paid_social_prospecting_reach','paid_social_remarketing_reach'],
    frequency_cols=["paid_social_prospecting_frequency",'paid_social_remarketing_frequency'],
    rf_spend_cols=["paid_social_prospecting_spend",'paid_social_remarketing_spend'],
    rf_channels=["paid_social_prospecting",'paid_social_remarketing'],
)

4. Finally, build the InputData.

In [ ]:
data = builder.build()

Note that the simulated data here contains reach and frequency channels. We recommend including reach and frequency data whenever they are available. For information about the advantages of utilizing reach and frequency, see [Bayesian Hierarchical Media Mix Model Incorporating Reach and Frequency Data](https://research.google/pubs/bayesian-hierarchical-media-mix-model-incorporating-reach-and-frequency-data/#:~:text=By%20incorporating%20R%26F%20into%20MMM,based%20on%20optimal%20frequency%20recommendations.).

<a name="configure-model"></a>
## Step 2: Configure the model

Meridian uses Bayesian framework and Markov Chain Monte Carlo (MCMC) algorithms to sample from the posterior distribution.

1\. Inititalize the `Meridian` class by passing the loaded data and the customized model specification. One advantage of Meridian lies in its capacity to calibrate the model directly through ROI priors, as described in [Media Mix Model Calibration With Bayesian Priors](https://research.google/pubs/media-mix-model-calibration-with-bayesian-priors/). In this particular example, the ROI priors for all media channels are identical, with each being represented as Lognormal(0.2, 0.9).

In [ ]:
import tensorflow_probability as tfp
from meridian.model import spec, prior_distribution
from meridian import constants  # if you’re using constants.ROI_M / ROI_RF

roi_mu = 2.284199793236827
roi_sigma = 0.7

roi_rf_mu = 2.3
roi_rf_sigma = 0.7

prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M),
    roi_rf=tfp.distributions.LogNormal(roi_rf_mu, roi_rf_sigma, name=constants.ROI_RF),
)

model_spec = spec.ModelSpec(
    prior=prior,
    media_prior_type="roi",   # optional but clearer (roi is default)
    enable_aks=True,
)

mmm = model.Meridian(input_data=data, model_spec=model_spec)


2\. Use the `sample_prior()` and `sample_posterior()` methods to obtain samples from the prior and posterior distributions of model parameters. If you are using the T4 GPU runtime this step may take about 10 minutes for the provided data set.

In [ ]:
%%time
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=1
)

For more information about configuring the parameters and using a customized model specification, such as setting different ROI priors for each media channel, see [Configure the model](https://developers.google.com/meridian/docs/user-guide/configure-model).

<a name="model-diagnostics"></a>
## Step 3: Run model diagnostics

After the model is built, you must assess convergence, debug the model if needed, and then assess the model fit.

1\. Assess convergence. Run the following code to generate r-hat statistics. R-hat close to 1.0 indicate convergence. R-hat < 1.2 indicates approximate convergence and is a reasonable threshold for many problems.

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

In [ ]:
from meridian.analysis.review import reviewer
reviewer.ModelReviewer(mmm).run()

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_prior_and_posterior_distribution()

2\. Assess the model's fit by comparing the expected sales against the actual sales.

In [ ]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

For more information and additional model diagnostics checks, see [Modeling diagnostics](https://developers.google.com/meridian/docs/user-guide/model-diagnostics).

<a name="generate-summary"></a>
## Step 4: Generate model results & two-page output

To export the two-page HTML summary output, initialize the `Summarizer` class with the model object. Then pass in the filename, filepath, start date, and end date to `output_model_results_summary` to run the summary for that time duration and save it to the specified file.

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
filepath = meridian_root
start_date = '2024-03-31'
end_date = '2025-08-31'
mmm_summarizer.output_model_results_summary(
    'summary_output_with_reach.html', filepath, start_date, end_date
)

Here is a preview of the two-page output based on the simulated data:

In [ ]:
IPython.display.HTML(filename='/content/drive/MyDrive/summary_output_with_reach.html')

For a customized two-page report, model results summary table, and individual visualizations, see [Model results report](https://developers.google.com/meridian/docs/user-guide/generate-model-results-report) and [plot media visualizations](https://developers.google.com/meridian/docs/user-guide/plot-media-visualizations).





<a name="generate-optimize"></a>
## Step 5: Run budget optimization & generate an optimization report

You can choose what scenario to run for the budget allocation. In default scenario, you find the optimal allocation across channels for a given budget to maximize the return on investment (ROI).

1\. Instantiate the `BudgetOptimizer` class and run the `optimize()` method without any customization, to run the default library's Fixed Budget Scenario to maximize ROI.

In [ ]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

2\. Export the 2-page HTML optimization report, which contains optimized spend allocations and ROI.

In [ ]:
filepath = '/content/drive/MyDrive'
optimization_results.output_optimization_summary(
    'optimization_output_reach.html', filepath
)

In [ ]:
IPython.display.HTML(filename='/content/drive/MyDrive/optimization_output.html')

For information about customized optimization scenarios, such as flexible budget scenarios, see [Budget optimization scenarios](https://developers.google.com/meridian/docs/user-guide/budget-optimization-scenarios). For more information about optimization results summary and individual visualizations, see [optimization results output](https://developers.google.com/meridian/docs/user-guide/generate-optimization-results-output) and [optimization visualizations](https://developers.google.com/meridian/docs/user-guide/plot-optimization-visualizations).

<a name="save-model"></a>
## Step 6: Save the model object

We recommend that you save the model object for future use. This helps you to  avoid repetitive model runs and saves time and computational resources. After the model object is saved, you can load it at a later stage to continue the analysis or visualizations without having to re-run the model.


Run the following codes to save the model object:

In [ ]:
file_path = '/content/drive/MyDrive/saved_mmm.pkl'
model.save_mmm(mmm, file_path)

Run the following codes to load the saved model:

In [ ]:
mmm = model.load_mmm(file_path)